# Model 3 — RAG (Retrieval-Augmented Generation)
**Team11 | WU LLM Course SS26**

Retrieves relevant Austrian statute passages via FAISS,
then generates answers with Gemini 2.5 Flash-Lite.

**Corpus:** EStG 1988, KStG, UStG 1994, GrEStG, BAO from RIS  
**Retriever:** paraphrase-multilingual-mpnet-base-v2 + FAISS  
**Generator:** Gemini 2.5 Flash-Lite (free tier)

In [ ]:
# Cell 1 — Install libraries
!pip install -q sentence-transformers faiss-cpu google-genai pandas requests beautifulsoup4

In [ ]:
# Cell 2 — Imports and config
import os, re, time, pickle
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import faiss
from sentence_transformers import SentenceTransformer
from google import genai

# Gemini API key — get free at https://aistudio.google.com/apikey
GEMINI_API_KEY = "YOUR_API_KEY"  # <-- paste your key here
client = genai.Client(api_key=GEMINI_API_KEY)

# cache directory
CACHE_DIR = "rag_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# shared prompt — same across all 3 models
SYSTEM_PROMPT = (
    "Beantworte die folgende Frage zum österreichischen Steuerrecht auf Deutsch. "
    "Antworte in maximal 1–4 Sätzen. "
    "Nenne die einschlägige Rechtsnorm (z.B. § 7 Abs 1 KStG). "
    "Halluziniere keine Paragraphen. Wenn unklar, formuliere vorsichtig."
)

## Corpus — Scrape Austrian laws from RIS

We fetch the full text of each law from `ris.bka.gv.at`, then split by § boundaries.

| Law | Gesetzesnummer | Question categories |
|-----|---------------|---------------------|
| EStG 1988 | 10004570 | EStG, ESTG-NEW, EMP, SELF, TAX-INTL |
| KStG 1988 | 10004569 | CORP-TAX |
| UStG 1994 | 10004873 | VAT-DOM, VAT-INTL |
| GrEStG 1987 | 10004531 | GRESt-AT |
| BAO | 10003940 | TAX-INTL (procedural) |

In [ ]:
# Cell 3 — Scrape laws from RIS or load from cache

LAWS = {
    "EStG 1988":  "10004570",
    "KStG 1988":  "10004569",
    "UStG 1994":  "10004873",
    "GrEStG 1987": "10004531",
    "BAO":        "10003940",
}

CHUNKS_PATH = os.path.join(CACHE_DIR, "chunks.pkl")

def scrape_law(name, gesetzesnummer):
    """Fetch full text of a law from RIS and split into § chunks."""
    url = f"https://www.ris.bka.gv.at/GeltendeFassung.wxe?Abfrage=Bundesnormen&Gesetzesnummer={gesetzesnummer}"
    print(f"  Fetching {name} from RIS...")
    resp = requests.get(url, timeout=60)
    resp.encoding = "utf-8"
    soup = BeautifulSoup(resp.text, "html.parser")

    # RIS renders law text in the main content area
    # extract all text, then split by § markers
    text = soup.get_text(separator="\n", strip=True)

    # split on § boundaries: "§ 1.", "§ 2.", "§ 10a." etc.
    parts = re.split(r'(?=§\s*\d+[a-z]?\.)', text)

    chunks = []
    for part in parts:
        part = part.strip()
        if not part or not part.startswith("§"):
            continue

        # extract section number: "§ 7." -> "§ 7"
        m = re.match(r'(§\s*\d+[a-z]?)\.?', part)
        section = m.group(1) if m else "?"

        # skip very short fragments (table of contents entries etc.)
        if len(part) < 50:
            continue

        chunks.append({
            "law": name,
            "section": section,
            "text": part[:2000],  # cap at 2000 chars (~400 tokens)
            "url": f"https://www.ris.bka.gv.at/NormDokument.wxe?Abfrage=Bundesnormen&Gesetzesnummer={gesetzesnummer}&Paragraf={section.replace('§ ', '').strip()}"
        })

    print(f"  → {name}: {len(chunks)} chunks")
    return chunks


# load from cache or scrape
if os.path.exists(CHUNKS_PATH):
    with open(CHUNKS_PATH, "rb") as f:
        all_chunks = pickle.load(f)
    print(f"Loaded {len(all_chunks)} chunks from cache.")
else:
    all_chunks = []
    for name, gnr in LAWS.items():
        try:
            all_chunks.extend(scrape_law(name, gnr))
        except Exception as e:
            print(f"  ERROR scraping {name}: {e} — skipping")
    # save cache
    with open(CHUNKS_PATH, "wb") as f:
        pickle.dump(all_chunks, f)
    print(f"\nTotal: {len(all_chunks)} chunks saved to cache.")

# show distribution
from collections import Counter
law_counts = Counter(c["law"] for c in all_chunks)
for law, count in law_counts.items():
    print(f"  {law}: {count} chunks")

## Build FAISS index

In [ ]:
# Cell 4 — Embed chunks + build FAISS index, or load from cache

EMBEDDINGS_PATH = os.path.join(CACHE_DIR, "embeddings.npy")
INDEX_PATH = os.path.join(CACHE_DIR, "index.faiss")

# multilingual model — better for Austrian legal German than English-only models
embed_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

if os.path.exists(EMBEDDINGS_PATH) and os.path.exists(INDEX_PATH):
    # load cached embeddings and index
    embeddings = np.load(EMBEDDINGS_PATH)
    index = faiss.read_index(INDEX_PATH)
    print(f"Loaded index with {index.ntotal} vectors from cache.")
else:
    # embed all chunks in one batch
    texts = [c["text"] for c in all_chunks]
    print(f"Embedding {len(texts)} chunks...")
    embeddings = embed_model.encode(texts, batch_size=64, show_progress_bar=True)
    embeddings = embeddings.astype("float32")

    # build FAISS index (inner product = cosine similarity on normalized vectors)
    faiss.normalize_L2(embeddings)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # inner product after L2 norm = cosine similarity
    index.add(embeddings)

    # save cache
    np.save(EMBEDDINGS_PATH, embeddings)
    faiss.write_index(index, INDEX_PATH)
    print(f"Built and cached FAISS index: {index.ntotal} vectors, dim={dim}")

## Mini retrieval test
Check that the correct §§ appear in top-3 before running the full pipeline.

In [ ]:
# Cell 5 — Mini retrieval test: 5 sample questions

TOP_K = 3  # default — adjust if mini-test shows we need more context

test_questions = [
    "Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?",           # expect KStG § 7
    "Welche Einkünfte unterliegen der unbeschränkten Steuerpflicht?",                    # expect EStG § 1
    "Welche Lieferungen und Leistungen unterliegen der Umsatzsteuer?",                   # expect UStG § 1
    "Was ist ein Grunderwerbsteuer-pflichtiger Vorgang?",                                # expect GrEStG § 1
    "Wie wird eine Dividende einer österreichischen Tochtergesellschaft behandelt?",     # expect KStG § 10
]

for q in test_questions:
    # embed the question
    q_vec = embed_model.encode([q]).astype("float32")
    faiss.normalize_L2(q_vec)

    # search
    scores, indices = index.search(q_vec, TOP_K)

    print(f"\nQ: {q[:80]}...")
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        chunk = all_chunks[idx]
        print(f"  [{rank+1}] {chunk['law']} {chunk['section']} (score={score:.3f})")
        print(f"      {chunk['text'][:100]}...")

## Full retrieval + generation

In [ ]:
# Cell 6 — Upload dataset_clean.csv (Colab)
from google.colab import files
uploaded = files.upload()  # upload dataset_clean.csv

In [ ]:
# Cell 7 — Batch-encode all questions + retrieve top-K passages

df = pd.read_csv("dataset_clean.csv")
print(f"Loaded {len(df)} questions.")

questions = df["prompt"].tolist()

# encode all questions at once
print("Encoding questions...")
q_embeddings = embed_model.encode(questions, batch_size=64, show_progress_bar=True).astype("float32")
faiss.normalize_L2(q_embeddings)

# retrieve top-K for all questions
print(f"Retrieving top-{TOP_K} passages per question...")
all_scores, all_indices = index.search(q_embeddings, TOP_K)

# build context strings for each question
passages_list = []
for i in range(len(questions)):
    context_parts = []
    for rank in range(TOP_K):
        idx = all_indices[i][rank]
        chunk = all_chunks[idx]
        # format: "[KStG 1988 § 7] text..."
        context_parts.append(f"[{chunk['law']} {chunk['section']}]\n{chunk['text']}")
    passages_list.append("\n\n".join(context_parts))

print(f"Done — {len(passages_list)} passage sets ready.")

In [ ]:
# Cell 8 — Generate answers with Gemini 2.5 Flash-Lite
# NOTE: billing must be enabled — free tier only 20 RPD (not enough for 643 questions)
# gemini-2.0-flash has limit=0 for some accounts — use gemini-2.5-flash-lite
CHECKPOINT = "checkpoint_model3.csv"
BATCH_SIZE = 10  # stay safely under 15 RPM limit

# load checkpoint if exists (resume from where we left off)
if os.path.exists(CHECKPOINT):
    done_df = pd.read_csv(CHECKPOINT)
    results = done_df.to_dict("records")
    done_ids = set(done_df["id"])
    print(f"Resuming: {len(results)} already done.")
else:
    results = []
    done_ids = set()

todo = [(i, row) for i, row in df.iterrows() if row["id"] not in done_ids]
print(f"{len(todo)} questions remaining.")

for batch_start in range(0, len(todo), BATCH_SIZE):
    batch = todo[batch_start:batch_start + BATCH_SIZE]

    for i, row in batch:
        prompt = (
            SYSTEM_PROMPT + "\n\n"
            "Hier sind relevante Gesetzestexte als Kontext:\n\n"
            + passages_list[i] + "\n\n"
            "Frage: " + row["prompt"]
        )

        answer = ""
        for attempt in range(3):
            try:
                resp = client.models.generate_content(
                    model="gemini-2.5-flash-lite",
                    contents=prompt
                )
                answer = resp.text.strip()
                break
            except Exception as e:
                wait = 15 * (attempt + 1)
                print(f"  Retry {attempt+1} for {row['id']}: waiting {wait}s...")
                time.sleep(wait)
                if attempt == 2:
                    print(f"  FAILED {row['id']}: {e}")
                    answer = "Fehler bei der Generierung."

        results.append({"id": row["id"], "answer": answer})

    # save checkpoint after every batch
    pd.DataFrame(results).to_csv(CHECKPOINT, index=False)
    print(f"  Checkpoint: {len(results)}/{len(df)} done")

    # wait 65s between batches to respect 15 RPM limit
    if batch_start + BATCH_SIZE < len(todo):
        print("  Waiting 65s...")
        time.sleep(65)

print(f"\nDone! {len(results)} answers generated.")

In [ ]:
# Cell 9 — Save final results to CSV
output_path = "model3_rag.csv"
result_df = pd.DataFrame(results)

# make sure rows are in the same order as the original dataset
result_df = result_df.set_index("id").loc[df["id"]].reset_index()
result_df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

# download
files.download(output_path)

In [ ]:
# Cell 10 — Validate submission format
ref = pd.read_csv("dataset_clean.csv")
sub = pd.read_csv(output_path)

assert list(sub.columns) == ["id", "answer"], "Wrong columns"
assert len(sub) == len(ref), f"Expected {len(ref)} rows, got {len(sub)}"
assert sub["id"].tolist() == ref["id"].tolist(), "IDs don't match"
assert sub["id"].nunique() == len(sub), "Duplicate IDs"
assert sub["answer"].notna().all(), "Null answers found"
assert (sub["answer"].astype(str).str.strip() != "").all(), "Blank answers found"
top_freq = sub["answer"].value_counts().iloc[0]
assert top_freq < 20, f"Repeated answer {top_freq}x — possible crash"

print(f"OK — {output_path} is valid ({len(sub)} rows)")
print("\nSample answers:")
for _, row in sub.head(3).iterrows():
    print(f"\n{row['id']}: {row['answer'][:150]}...")